# DeepEval MCP Metrics - Excel Trace Evaluation

This notebook only evaluates MCP trace data from an Excel workbook. The MCP server and trace generator run outside this notebook. The Excel file is the handoff artifact.

## 1. Install Dependencies

In [ ]:
%pip install deepeval==4.0.7 mcp==1.28.1 openai==1.99.5 pandas==3.0.3 openpyxl==3.1.5


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.1/413.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninst

## 2. Imports

In [ ]:
from pathlib import Path

import pandas as pd

from deepeval.metrics.mcp.mcp_task_completion import MCPTaskCompletionMetric
from deepeval.metrics.mcp.multi_turn_mcp_use_metric import MultiTurnMCPUseMetric
from deepeval.metrics.mcp_use_metric.mcp_use_metric import MCPUseMetric
from deepeval.models.llms.azure_model import AzureOpenAIModel
from deepeval.test_case import ConversationalTestCase, LLMTestCase, MCPServer, MCPToolCall, Turn
from mcp.types import CallToolResult, TextContent, Tool

## 3. Judge Model Credentials

Set the judge model credentials before reading or evaluating the Excel trace.

In [ ]:
from urllib.parse import urlparse

# Fill these before running metric cells. Do not leave any PASTE_* placeholders.
AZURE_OPENAI_API_KEY = "FkrroqazToY6a2FXJ3w3AAABACOGKkmt"
AZURE_OPENAI_ENDPOINT = "https://ai-a.openai.azure.com/"  # Example: https://your-resource.openai.azure.com
AZURE_OPENAI_DEPLOYMENT_NAME = "gpt-5"
AZURE_OPENAI_MODEL_NAME = "gpt-5"
AZURE_OPENAI_API_VERSION = "2024-view"

AZURE_OPENAI_COST_PER_INPUT_TOKEN = 0.0
AZURE_OPENAI_COST_PER_OUTPUT_TOKEN = 0.0


def validate_azure_openai_credentials():
    values = {
        "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
        "AZURE_OPENAI_DEPLOYMENT_NAME": AZURE_OPENAI_DEPLOYMENT_NAME,
        "AZURE_OPENAI_API_VERSION": AZURE_OPENAI_API_VERSION,
    }
    missing_or_placeholder = [
        name
        for name, value in values.items()
        if not str(value).strip() or "PASTE_" in str(value)
    ]
    if missing_or_placeholder:
        raise ValueError(
            "Set real Azure OpenAI values before running evaluation metrics. Missing or placeholder fields: "
            + ", ".join(missing_or_placeholder)
        )

    endpoint = AZURE_OPENAI_ENDPOINT.strip().rstrip("/")
    parsed = urlparse(endpoint)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError(
            "AZURE_OPENAI_ENDPOINT must be a full HTTPS URL, for example: "
            "https://your-resource.openai.azure.com"
        )
    if " " in parsed.netloc or "PASTE" in parsed.netloc:
        raise ValueError(f"AZURE_OPENAI_ENDPOINT host is invalid: {parsed.netloc!r}")
    return endpoint


AZURE_OPENAI_ENDPOINT = validate_azure_openai_credentials()

judge_model = AzureOpenAIModel(
    model=AZURE_OPENAI_MODEL_NAME,
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    cost_per_input_token=AZURE_OPENAI_COST_PER_INPUT_TOKEN,
    cost_per_output_token=AZURE_OPENAI_COST_PER_OUTPUT_TOKEN,
)


## 4. Excel File

Run the next cell to upload `Live_Dummy_MCP_Traces.xlsx` in Colab. If you are running locally and the file is already beside this notebook, the same cell will use that local file instead.

In [ ]:
TRACE_FILE = Path("Live_Dummy_MCP_Traces.xlsx")

try:
    from google.colab import files

    print("Upload Live_Dummy_MCP_Traces.xlsx")
    uploaded = files.upload()

    if uploaded:
        uploaded_name = next(iter(uploaded.keys()))
        TRACE_FILE = Path(uploaded_name)
except ImportError:
    print("Not running in Colab. Using local Excel file path.")

if not TRACE_FILE.exists():
    raise FileNotFoundError(
        "Live_Dummy_MCP_Traces.xlsx was not found. Upload the generated Excel file or place it beside this notebook."
    )

TRACE_FILE

Upload Live_Dummy_MCP_Traces.xlsx


Saving Live_Dummy_MCP_Traces.xlsx to Live_Dummy_MCP_Traces.xlsx


PosixPath('Live_Dummy_MCP_Traces.xlsx')

## 5. Read Generated Sheets

In [ ]:
input_cases = pd.read_excel(TRACE_FILE, sheet_name="input_cases")
tools_definition = pd.read_excel(TRACE_FILE, sheet_name="tools_definition")
deepeval_objects = pd.read_excel(TRACE_FILE, sheet_name="deepeval_objects")
support_data = pd.read_excel(TRACE_FILE, sheet_name="support_data")

required_input_columns = {"case_id", "expected_tools_called", "actual_tools_called", "run_status"}
missing_columns = required_input_columns - set(input_cases.columns)
if missing_columns:
    raise ValueError(f"input_cases sheet is missing columns: {sorted(missing_columns)}")

required_deepeval_columns = {
    "case_id",
    "dummy_mcp_server_object",
    "llm_test_case_object",
    "conversational_test_case_object",
}
missing_deepeval_columns = required_deepeval_columns - set(deepeval_objects.columns)
if missing_deepeval_columns:
    raise ValueError(f"deepeval_objects sheet is missing columns: {sorted(missing_deepeval_columns)}")

input_cases


,case_id,case_type,run_at,input,expected_output,expected_tools_called,actual_output,actual_tools_called,tool_call_index,tool_name,mcp_jsonrpc_request_json,mcp_jsonrpc_response_json,tool_arguments_json,tool_result_json,run_status
0,CASE-002-20260715201406,single_turn,2026-07-15T20:14:06,Customer C-1003 wants to know whether order OR...,"Inspect ORD-9003, review refund policy, calcul...","get_order_status,search_policy_docs,calculate_...",Here’s what I found for order ORD-9003 (Smartw...,"get_order_status,search_policy_docs",1,get_order_status,"{\n ""jsonrpc"": ""2.0"",\n ""id"": 1,\n ""method""...","{\n ""jsonrpc"": ""2.0"",\n ""id"": 1,\n ""result""...","{\n ""order_id"": ""ORD-9003""\n}","{\n ""order_id"": ""ORD-9003"",\n ""customer_id"":...",completed
1,CASE-002-20260715201406,single_turn,2026-07-15T20:14:06,Customer C-1003 wants to know whether order OR...,"Inspect ORD-9003, review refund policy, calcul...","get_order_status,search_policy_docs,calculate_...",Here’s what I found for order ORD-9003 (Smartw...,"get_order_status,search_policy_docs",2,search_policy_docs,"{\n ""jsonrpc"": ""2.0"",\n ""id"": 2,\n ""method""...","{\n ""jsonrpc"": ""2.0"",\n ""id"": 2,\n ""result""...","{\n ""query"": ""refund policy return window del...","{\n ""query"": ""refund policy return window del...",completed
2,CASE-008-20260715201443,single_turn,2026-07-15T20:14:43,Customer C-1004 is travelling soon and asks wh...,"Look up C-1004, inspect ORD-9004, and share th...","get_customer_profile,get_order_status",I checked your order. The ergonomic keyboard (...,"get_customer_profile,get_order_status",1,get_customer_profile,"{\n ""jsonrpc"": ""2.0"",\n ""id"": 1,\n ""method""...","{\n ""jsonrpc"": ""2.0"",\n ""id"": 1,\n ""result""...","{\n ""customer_id"": ""C-1004""\n}","{\n ""customer_id"": ""C-1004"",\n ""name"": ""Dev ...",completed
3,CASE-008-20260715201443,single_turn,2026-07-15T20:14:43,Customer C-1004 is travelling soon and asks wh...,"Look up C-1004, inspect ORD-9004, and share th...","get_customer_profile,get_order_status",I checked your order. The ergonomic keyboard (...,"get_customer_profile,get_order_status",2,get_order_status,"{\n ""jsonrpc"": ""2.0"",\n ""id"": 2,\n ""method""...","{\n ""jsonrpc"": ""2.0"",\n ""id"": 2,\n ""result""...","{\n ""order_id"": ""ORD-9004""\n}","{\n ""order_id"": ""ORD-9004"",\n ""customer_id"":...",completed


## 6. Inspect Tool Definitions

In [ ]:
pd.set_option("display.max_colwidth", 3000)

tools_definition[["tool_name", "description", "deepeval_tool_object"]]


,tool_name,description,deepeval_tool_object
0,get_customer_profile,Get a customer profile by customer ID.,"Tool(\n name='get_customer_profile',\n description='Get a customer profile by customer ID.',\n inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'Customer ID such as C-1002.'}}, 'required': ['customer_id'], 'additionalProperties': False},\n)"
1,get_order_status,Get order status and order details by order ID.,"Tool(\n name='get_order_status',\n description='Get order status and order details by order ID.',\n inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': 'Order ID such as ORD-9002.'}}, 'required': ['order_id'], 'additionalProperties': False},\n)"
2,search_policy_docs,Search support policy documents.,"Tool(\n name='search_policy_docs',\n description='Search support policy documents.',\n inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Policy search query.'}}, 'required': ['query'], 'additionalProperties': False},\n)"
3,calculate_refund,Calculate refund eligibility for an order.,"Tool(\n name='calculate_refund',\n description='Calculate refund eligibility for an order.',\n inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 'required': ['order_id', 'reason'], 'additionalProperties': False},\n)"
4,create_support_ticket,Create a support ticket for a customer issue.,"Tool(\n name='create_support_ticket',\n description='Create a support ticket for a customer issue.',\n inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string'}, 'issue_summary': {'type': 'string'}, 'priority': {'type': 'string', 'enum': ['low', 'normal', 'high']}}, 'required': ['customer_id', 'issue_summary', 'priority'], 'additionalProperties': False},\n)"


## 7. Inspect Generated DeepEval Objects

In [ ]:
deepeval_objects[["case_id", "dummy_mcp_server_object", "llm_test_case_object", "conversational_test_case_object"]]


case_id  \
0  CASE-002-20260715201406   
1  CASE-008-20260715201443   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  dummy_mcp_server_object  \
0  dummy_support_server = MCPServer(\n    server_name="dummy_support",\n    transport="streamable-http",\n    available_tools=[\n        Tool(\n            name='get_customer_profile',\n            description='Get a customer profile by customer ID.',\n            inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'Customer ID such as C-1002.'}}, 'required': ['customer_id'], 'additionalProperties': False},\n        ),\n        Tool(\n            name='get_order_status',\n            description='Get order status and order details by order ID.',\n            inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': 'Order ID such as ORD-9002.'}}, 'required': ['order_id'], 'additionalProperties': False},\n        ),\n        Tool(\n            name='search_policy_docs',\n            description='Search support policy documents.',\n            inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Policy search query.'}}, 'required': ['query'], 'additionalProperties': False},\n        ),\n        Tool(\n            name='calculate_refund',\n            description='Calculate refund eligibility for an order.',\n            inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 'required': ['order_id', 'reason'], 'additionalProperties': False},\n        ),\n        Tool(\n            name='create_support_ticket',\n            description='Create a support ticket for a customer issue.',\n            inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string'}, 'issue_summary': {'type': 'string'}, 'priority': {'type': 'string', 'enum': ['low', 'normal', 'high']}}, 'required': ['customer_id', 'issue_summary', 'priority'], 'additionalProperties': False},\n        )\n    ],\n)   
1  dummy_support_server = MCPServer(\n    server_name="dummy_support",\n    transport="streamable-http",\n    available_tools=[\n        Tool(\n            na

## 8. Load MCP Server Object From Excel

In [ ]:
server_source = deepeval_objects["dummy_mcp_server_object"].dropna().astype(str).str.strip()
if server_source.empty:
    raise ValueError("deepeval_objects has no dummy_mcp_server_object snippet to load.")

server_namespace = {"MCPServer": MCPServer, "Tool": Tool}
exec(server_source.iloc[0], {"__builtins__": {}, **server_namespace}, server_namespace)
dummy_support_server = server_namespace["dummy_support_server"]

dummy_support_server


MCPServer(server_name='dummy_support', transport='streamable-http', available_tools=[Tool(name='get_customer_profile', title=None, description='Get a customer profile by customer ID.', inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'Customer ID such as C-1002.'}}, 'required': ['customer_id'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='get_order_status', title=None, description='Get order status and order details by order ID.', inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': 'Order ID such as ORD-9002.'}}, 'required': ['order_id'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='search_policy_docs', title=None, description='Search support policy documents.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Pol

## 9. Test Case Shapes

`MCPUseMetric` evaluates single-turn `LLMTestCase` objects. `MultiTurnMCPUseMetric` and `MCPTaskCompletionMetric` evaluate the `ConversationalTestCase` objects already written in the Excel trace.

The notebook uses the syntaxed DeepEval objects from the `deepeval_objects` sheet directly.

## 10. Use DeepEval Test Cases From Excel

In [ ]:
def first_non_empty(series):
    for value in series:
        if pd.notna(value) and str(value).strip():
            return str(value)
    return ""


def expected_label_from_status(run_status):
    run_status = str(run_status).strip()
    if run_status and run_status not in {"completed", "step_limit"}:
        return run_status
    return "Positive"


def evaluate_deepeval_object(source, case_id, column_name):
    if pd.isna(source) or not str(source).strip():
        raise ValueError(f"Missing {column_name} for case {case_id}")
    try:
        return eval(str(source), deepeval_eval_globals, {})
    except Exception as exc:
        raise ValueError(f"Could not load {column_name} for case {case_id}: {exc}") from exc


import json


def mcp_tool_result_value(tool):
    structured_content = getattr(tool.result, "structuredContent", None) or {}
    if isinstance(structured_content, dict) and "result" in structured_content:
        return structured_content["result"]
    if structured_content:
        return structured_content

    for content_item in getattr(tool.result, "content", []) or []:
        text = getattr(content_item, "text", "")
        if not str(text).strip():
            continue
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text

    return {}


from deepeval.metrics.mcp.schema import Task


def get_tasks_from_excel_conversations(self, unit_interactions):
    tasks = []

    for unit_interaction in unit_interactions:
        if len(unit_interaction) < 2:
            continue

        # Ignore closing exchanges like: user "Thank you" -> assistant closing.
        # Keep only units where the assistant actually made MCP calls.
        # if not any(turn._mcp_interaction for turn in unit_interaction[1:]):
        #     continue

        user_messages = ""
        for turn in unit_interaction:
            if turn.role == "user":
                user_messages += turn.content + "\n"
            else:
                break

        new_task = Task(task=user_messages, steps_taken=[])

        for turn in unit_interaction[1:]:
            if turn._mcp_interaction:
                mcp_interaction = "Tools called by agent: \n"

                if turn.mcp_tools_called is not None:
                    for tool in turn.mcp_tools_called:
                        mcp_interaction += (
                            f"\n<Tool Called>\n"
                            f"\n**This does not appear to user**\n"
                            f"Name: {tool.name}\n"
                            f"Args: {tool.args}\n"
                            f"Result: \n{mcp_tool_result_value(tool)}\n"
                            f"</Tool Called>\n"
                        )

                new_task.steps_taken.append(mcp_interaction)

                if str(turn.content).strip():
                    new_task.steps_taken.append("Agent's response to user: \n" + turn.content)
            else:
                new_task.steps_taken.append("Agent's response to user: \n" + turn.content)

        tasks.append(new_task)

    return tasks


MultiTurnMCPUseMetric._get_tasks = get_tasks_from_excel_conversations
MCPTaskCompletionMetric._get_tasks = get_tasks_from_excel_conversations


input_case_metadata = []
for case_id, case_rows in input_cases.fillna("").groupby("case_id", sort=False):
    input_case_metadata.append(
        {
            "case_id": str(case_id),
            "expected_label": expected_label_from_status(first_non_empty(case_rows["run_status"])),
            "expected_tools_called": first_non_empty(case_rows["expected_tools_called"]),
            "actual_tools_called": first_non_empty(case_rows["actual_tools_called"]),
        }
    )

case_metadata_by_id = {row["case_id"]: row for row in input_case_metadata}

deepeval_eval_globals = {
    "__builtins__": {},
    "LLMTestCase": LLMTestCase,
    "ConversationalTestCase": ConversationalTestCase,
    "MCPServer": MCPServer,
    "MCPToolCall": MCPToolCall,
    "Turn": Turn,
    "CallToolResult": CallToolResult,
    "TextContent": TextContent,
    "Tool": Tool,
    "dummy_support_server": dummy_support_server,
}

test_cases = []
conversational_test_cases = []
evaluation_case_rows = []

for _, object_row in deepeval_objects.fillna("").iterrows():
    case_id = str(object_row["case_id"]).strip()
    if not case_id:
        continue

    llm_test_case = evaluate_deepeval_object(
        object_row["llm_test_case_object"],
        case_id,
        "llm_test_case_object",
    )
    conversational_test_case = evaluate_deepeval_object(
        object_row["conversational_test_case_object"],
        case_id,
        "conversational_test_case_object",
    )

    test_cases.append(llm_test_case)
    conversational_test_cases.append(conversational_test_case)

    metadata = case_metadata_by_id.get(case_id, {})
    evaluation_case_rows.append(
        {
            "case_id": case_id,
            "expected_label": metadata.get("expected_label", "Positive"),
            "tool_call_count": len(llm_test_case.mcp_tools_called or []),
            "expected_tools_called": metadata.get("expected_tools_called", ""),
            "actual_tools_called": metadata.get("actual_tools_called", ""),
        }
    )

from deepeval.metrics.utils import get_unit_interactions

conversation_debug_rows = []
for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    unit_interactions = get_unit_interactions(conversational_case.turns)
    metric_tasks = get_tasks_from_excel_conversations(None, unit_interactions)

    conversation_debug_rows.append(
        {
            "case_id": case_row["case_id"],
            "turn_count": len(conversational_case.turns),
            "unit_interaction_count": len(unit_interactions),
            "metric_task_count": len(metric_tasks),
            "tool_call_count": case_row["tool_call_count"],
        }
    )

conversation_debug = pd.DataFrame(conversation_debug_rows)
empty_task_cases = conversation_debug[conversation_debug["metric_task_count"] == 0]

if not empty_task_cases.empty:
    raise ValueError(
        "Multi-turn metrics would extract empty Tasks for these cases: "
        + ", ".join(empty_task_cases["case_id"].astype(str).tolist())
    )

len(test_cases), len(conversational_test_cases), pd.DataFrame(evaluation_case_rows), conversation_debug

(2,
 2,
                    case_id expected_label  tool_call_count  \
 0  CASE-002-20260715201406       Positive                2   
 1  CASE-008-20260715201443       Positive                2   
 
                                   expected_tools_called  \
 0  get_order_status,search_policy_docs,calculate_refund   
 1                 get_customer_profile,get_order_status   
 
                      actual_tools_called  
 0    get_order_status,search_policy_docs  
 1  get_customer_profile,get_order_status  ,
                    case_id  turn_count  unit_interaction_count  \
 0  CASE-002-20260715201406           4                       2   
 1  CASE-008-20260715201443           4                       2   
 
    metric_task_count  tool_call_count  
 0                  2                2  
 1                  2                2  )

## 11. Run MCPUseMetric

In [ ]:
mcp_use_rows = []

for test_case, case_row in zip(test_cases, evaluation_case_rows):
    metric = MCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(test_case)
    mcp_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_use_report = pd.DataFrame(mcp_use_rows)
mcp_use_report

Output()

**************************************************

MCP Use Verbose Logs

**************************************************

MCP Primitives Available: 
MCP Server dummy_support

Available Tools:
[
    Tool(name='get_customer_profile', title=None, description='Get a customer profile by customer ID.', 
inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'Customer ID such as
C-1002.'}}, 'required': ['customer_id'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='get_order_status', title=None, description='Get order status and order details by order ID.', 
inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': 'Order ID such as 
ORD-9002.'}}, 'required': ['order_id'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='search_policy_docs', title=None, description='Search support policy documents.', 
inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Policy search query.'}}, 
'required': ['query'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, meta=None, 
execution=None),
    Tool(name='calculate_refund', title=None, description='Calculate refund eligibility for an order.', 
inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 
'required': ['order_id', 'reason'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='create_support_ticket', title=None, description='Create a support ticket for a customer issue.', 
inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string'}, 'issue_summary': {'type': 
'string'}, 'priority': {'type': 'string', 'enum': ['low', 'normal', 'high']}}, 'required': ['customer_id', 
'issue_summary', 'priority'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, 
meta=None, execution=None)
] 
 
MCP Primitives Used: 

MCP Tools Called:
[
    MCPToolCall(name='get_order_status', args={'order_id': 'ORD-9003'}, result=CallToolResult(meta=None, 
content=[TextContent(type='text', text='{"order_id": "ORD-9003", "customer_id": "C-1003", "status": "delivered", 
"item": "Smartwatch", "amount": 149.0, "delivered_days_ago": 42, "issue": "Customer asked about refund 
eligibility."}', annotations=None, meta=None)], structuredContent=None, isError=False)),
    MCPToolCall(name='search_policy_docs', args={'query': 'refund policy return window delivered days eligibility 
delivered_days_ago 42 smartwatch'}, result=CallToolResult(meta=None, content=[TextContent(type='text', 
text='{"query": "refund policy return window delivered days eligibility delivered_days_ago 42 smartwatch", 
"matches": [{"policy_id": "refund", "title": "Refund policy", "summary": "Most delivered items are refundable 
within 30 days. Damaged items can be escalated after 30 days for manual review."}, {"policy_id": "delivery", 
"title": "Delivery delay policy", "summary": "If an order is delayed, share the reason and estimated delivery date.
Shipping fees may be refunded for delays over 5 days."}, {"policy_id": "damaged_item", "title": "Damaged item 
policy", "summary": "Damaged delivered items should be reviewed with order details. Create a high-priority ticket 
for expensive or urgent damaged-item cases."}, {"policy_id": "cancellation", "title": "Cancellation policy", 
"summary": "Cancelled orders should be explained using the cancellation reason. Ask the customer to retry payment 
or contact billing when payment authorization fails."}, {"policy_id": "shipping_status", "title": "Shipping status 
policy", "summary": "For shipped or processing orders, share the current status and estimated delivery date when 
available."}]}', annotations=None, meta=None)], structuredContent=None, isError=False))
] 
 
Primitive Usage Score: 0.75 
 
Primitive Usage Reason: The agent appropriately used get_order_status to verify 

...

MCP Use Verbose Logs

**************************************************

MCP Primitives Available: 
MCP Server dummy_support

Available Tools:
[
    Tool(name='get_customer_profile', title=None, description='Get a customer profile by customer ID.', 
inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'Customer ID such as
C-1002.'}}, 'required': ['customer_id'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='get_order_status', title=None, description='Get order status and order details by order ID.', 
inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string', 'description': 'Order ID such as 
ORD-9002.'}}, 'required': ['order_id'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='search_policy_docs', title=None, description='Search support policy documents.', 
inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Policy search query.'}}, 
'required': ['query'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, meta=None, 
execution=None),
    Tool(name='calculate_refund', title=None, description='Calculate refund eligibility for an order.', 
inputSchema={'type': 'object', 'properties': {'order_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 
'required': ['order_id', 'reason'], 'additionalProperties': False}, outputSchema=None, icons=None, 
annotations=None, meta=None, execution=None),
    Tool(name='create_support_ticket', title=None, description='Create a support ticket for a customer issue.', 
inputSchema={'type': 'object', 'properties': {'customer_id': {'type': 'string'}, 'issue_summary': {'type': 
'string'}, 'priority': {'type': 'string', 'enum': ['low', 'normal', 'high']}}, 'required': ['customer_id', 
'issue_summary', 'priority'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=None, 
meta=None, execution=None)
] 
 
MCP Primitives Used: 

MCP Tools Called:
[
    MCPToolCall(name='get_customer_profile', args={'customer_id': 'C-1004'}, result=CallToolResult(meta=None, 
content=[TextContent(type='text', text='{"customer_id": "C-1004", "name": "Dev Kapoor", "tier": "Gold", 
"primary_order_id": "ORD-9004", "email": "dev@example.com"}', annotations=None, meta=None)], 
structuredContent=None, isError=False)),
    MCPToolCall(name='get_order_status', args={'order_id': 'ORD-9004'}, result=CallToolResult(meta=None, 
content=[TextContent(type='text', text='{"order_id": "ORD-9004", "customer_id": "C-1004", "status": "processing", 
"item": "Ergonomic keyboard", "amount": 89.0, "estimated_delivery": "2026-07-16", "issue": "Customer wants delivery
expectations before travel."}', annotations=None, meta=None)], structuredContent=None, isError=False))
] 
 
Primitive Usage Score: 1.0 
 
Primitive Usage Reason: The agent appropriately used get_customer_profile to identify the customer’s relevant order
(primary_order_id) and then used get_order_status to retrieve the delivery estimate for the ergonomic keyboard. 
These are the correct tools for determining delivery timing given only a customer ID and item reference. No 
unnecessary tools were invoked, and no more suitable tool was available for this task. 
 
Argument Correctness Score: 1.0
 
Argument Correctness Reason: Both tool calls used correct argument structures and types with required fields 
present and no extras. get_customer_profile received {'customer_id':'C-1004'} (string as required), and 
get_order_status received {'order_id':'ORD-9004'} (string as required). The contents are semantically appropriate 
for determining the delivery date for the ergonomic keyboard referenced by the customer.

======================================================================

,case_id,expected_label,metric,score,success,reason
0,CASE-002-20260715201406,Positive,MCPUseMetric,0.75,True,"[\n\tThe agent appropriately used get_order_status to verify delivery timing and search_policy_docs to confirm the refund window and exceptions, which are relevant tools for assessing refund eligibility. They avoided unnecessary calls like get_customer_profile and did not prematurely create a ticket, which was appropriate. However, a dedicated calculate_refund tool existed specifically for determining refund eligibility by order, and using it (with a standard reason such as a normal return) could have provided a more direct, authoritative determination. Thus, the chosen tools were relevant but slightly suboptimal given the available primitives.\n\tAll tool calls used correctly formed arguments aligned with the schemas and task. get_order_status received the required string field order_id with value 'ORD-9003' and no extra properties. search_policy_docs received the required string field query; the free-text query included relevant terms about refund policy and return window; while it included case-specific tokens ('delivered_days_ago 42', 'smartwatch'), these are acceptable within a free-text search and not malformed. No required fields were missing, and no disallowed fields were included.\n]\n"
1,CASE-008-20260715201443,Positive,MCPUseMetric,1.00,True,"[\n\tThe agent appropriately used get_customer_profile to identify the customer’s relevant order (primary_order_id) and then used get_order_status to retrieve the delivery estimate for the ergonomic keyboard. These are the correct tools for determining delivery timing given only a customer ID and item reference. No unnecessary tools were invoked, and no more suitable tool was available for this task.\n\tBoth tool calls used correct argument structures and types with required fields present and no extras. get_customer_profile received {'customer_id':'C-1004'} (string as required), and get_order_status received {'order_id':'ORD-9004'} (string as required). The contents are semantically appropriate for determining the delivery date for the ergonomic keyboard referenced by the customer.\n]\n"


## 12. Run MultiTurnMCPUseMetric

In [ ]:
multi_turn_use_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MultiTurnMCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    multi_turn_use_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MultiTurnMCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

multi_turn_use_report = pd.DataFrame(multi_turn_use_rows)
multi_turn_use_report

Output()

**************************************************

Multi-Turn MCP Use Verbose Logs

**************************************************

Tasks:
[
    {
        "task": "Customer C-1003 wants to know whether order ORD-9003 can be refunded.\n",
        "steps_taken": [
            "Tools called by agent: \n\n<Tool Called>\n\n**This does not appear to user**\nName: 
get_order_status\nArgs: {'order_id': 'ORD-9003'}\nResult: \n{'order_id': 'ORD-9003', 'customer_id': 'C-1003', 
'status': 'delivered', 'item': 'Smartwatch', 'amount': 149.0, 'delivered_days_ago': 42, 'issue': 'Customer asked 
about refund eligibility.'}\n</Tool Called>\n\n<Tool Called>\n\n**This does not appear to user**\nName: 
search_policy_docs\nArgs: {'query': 'refund policy return window delivered days eligibility delivered_days_ago 42 
smartwatch'}\nResult: \n{'query': 'refund policy return window delivered days eligibility delivered_days_ago 42 
smartwatch', 'matches': [{'policy_id': 'refund', 'title': 'Refund policy', 'summary': 'Most delivered items are 
refundable within 30 days. Damaged items can be escalated after 30 days for manual review.'}, {'policy_id': 
'delivery', 'title': 'Delivery delay policy', 'summary': 'If an order is delayed, share the reason and estimated 
delivery date. Shipping fees may be refunded for delays over 5 days.'}, {'policy_id': 'damaged_item', 'title': 
'Damaged item policy', 'summary': 'Damaged delivered items should be reviewed with order details. Create a 
high-priority ticket for expensive or urgent damaged-item cases.'}, {'policy_id': 'cancellation', 'title': 
'Cancellation policy', 'summary': 'Cancelled orders should be explained using the cancellation reason. Ask the 
customer to retry payment or contact billing when payment authorization fails.'}, {'policy_id': 'shipping_status', 
'title': 'Shipping status policy', 'summary': 'For shipped or processing orders, share the current status and 
estimated delivery date when available.'}]}\n</Tool Called>\n",
            "Agent's response to user: \nHere\u2019s what I found for order ORD-9003 (Smartwatch) for customer 
C-1003: it was delivered 42 days ago. Our refund policy allows most delivered items to be refunded within 30 days. 
Since this is beyond 30 days, it isn\u2019t eligible for a standard refund.\n\nIf the smartwatch arrived damaged or
defective, we can submit it for a manual review even after 30 days. If that\u2019s the case, please share a brief 
description of the issue and a few photos showing the problem, and I can open a high-priority ticket for 
you.\n\nNext best step: let me know if the item is damaged/defective and if you\u2019d like me to create a review 
ticket. If not, I\u2019m afraid we won\u2019t be able to process a refund for this order."
        ]
    },
    {
        "task": "Thank you.\n",
        "steps_taken": [
            "Agent's response to user: \nYou are welcome. I hope the information provided was helpful."
        ]
    }
] 
 
Individual Scores & Reasons for Primitives:
[
    (0.75, 'The agent correctly used get_order_status and consulted policy docs, but it missed the more specialized
calculate_refund tool designed to determine eligibility. It also could have first prompted for the refund reason to
feed into calculate_refund for a definitive answer, reducing reliance on manual policy interpretation.'),
    (1.0, 'The user only expressed gratitude; no data retrieval or support actions were required, so not invoking 
any tools was optimal. The agent provided a polite acknowledgment without unnecessary tool usage.')
] 
 
Individual Scores & Reasons for Arguments:
[
    (1.0, 'Both tool calls matched their schemas: get_order_status provided a valid string order_id, and 
search_policy_docs provided a valid string query. No required fields were missing and no extra properties were 
included.'),
    (1.0, 'No tools or resources were invoked, so there were no arguments to evaluate against schemas. Nothing was 
misformatted or missing since no input schemas were engaged.')
]
 
Score: 0.875

...

Multi-Turn MCP Use Verbose Logs

**************************************************

Tasks:
[
    {
        "task": "Customer C-1004 is travelling soon and asks when the ergonomic keyboard will arrive.\n",
        "steps_taken": [
            "Tools called by agent: \n\n<Tool Called>\n\n**This does not appear to user**\nName: 
get_customer_profile\nArgs: {'customer_id': 'C-1004'}\nResult: \n{'customer_id': 'C-1004', 'name': 'Dev Kapoor', 
'tier': 'Gold', 'primary_order_id': 'ORD-9004', 'email': 'dev@example.com'}\n</Tool Called>\n\n<Tool 
Called>\n\n**This does not appear to user**\nName: get_order_status\nArgs: {'order_id': 'ORD-9004'}\nResult: 
\n{'order_id': 'ORD-9004', 'customer_id': 'C-1004', 'status': 'processing', 'item': 'Ergonomic keyboard', 'amount':
89.0, 'estimated_delivery': '2026-07-16', 'issue': 'Customer wants delivery expectations before travel.'}\n</Tool 
Called>\n",
            "Agent's response to user: \nI checked your order. The ergonomic keyboard (order ORD-9004) is currently
processing and is estimated to be delivered on 2026-07-16.\n\nIf that timing conflicts with your travel, I can look
into available options (such as updating the delivery address or arranging a carrier hold, if eligible). Next best 
step: let me know your departure date and whether you want me to check delivery-change options or create a support 
ticket on your behalf."
        ]
    },
    {
        "task": "Thank you.\n",
        "steps_taken": [
            "Agent's response to user: \nYou are welcome. I hope the information provided was helpful."
        ]
    }
] 
 
Individual Scores & Reasons for Primitives:
[
    (1.0, 'The agent correctly used get_customer_profile to obtain the order ID and get_order_status to retrieve 
the delivery estimate, which directly addressed the user’s question. No unnecessary tools were used, and there were
no better-suited tools for this task; the follow-up prompt about options was appropriate.'),
    (1.0, 'The user only expressed gratitude, requiring no data lookup or actions; no tools were necessary. The 
agent responded politely without unnecessary tool usage, which was optimal given the available tools and the 
task.')
] 
 
Individual Scores & Reasons for Arguments:
[
    (1.0, "Both tool calls matched their schemas: get_customer_profile received a valid 'customer_id' string, and 
get_order_status received a valid 'order_id' string. No required fields were missing and no extraneous parameters 
were included."),
    (1.0, 'No tools or resources were invoked, so no arguments were provided. Therefore, there were no schema 
violations or misformatted inputs to assess.')
]
 
Score: 1.0

======================================================================

,case_id,expected_label,metric,score,success,reason
0,CASE-002-20260715201406,Positive,MultiTurnMCPUseMetric,0.875,True,"The score is 0.875 because the agent generally selected and executed tools appropriately, provided schema-valid arguments, and avoided unnecessary calls when no action was required; the main deficiency was in a refund scenario where it did not use the specialized calculate_refund tool or elicit the refund reason needed for a definitive determination, relying instead on manual policy interpretation. Given the strong overall adherence to tool usage and argument schemas across interactions, the application meets expectations and the metric passes despite this isolated gap."
1,CASE-008-20260715201443,Positive,MultiTurnMCPUseMetric,1.000,True,"The score is 1.0 because the application consistently selected and executed the appropriate tools with correct, schema-compliant arguments when needed and refrained from unnecessary calls when a simple acknowledgment sufficed, directly addressing the user’s request by obtaining the order ID and delivery estimate without superfluous resources or parameters; this demonstrates accurate tool usage, parameter handling, and context-sensitive behavior, so the metric passed."


## 13. Run MCPTaskCompletionMetric

In [ ]:
task_completion_rows = []

for conversational_case, case_row in zip(conversational_test_cases, evaluation_case_rows):
    metric = MCPTaskCompletionMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(conversational_case)
    task_completion_rows.append(
        {
            "case_id": case_row["case_id"],
            "expected_label": case_row["expected_label"],
            "metric": "MCPTaskCompletionMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_task_completion_report = pd.DataFrame(task_completion_rows)
mcp_task_completion_report

Output()

**************************************************

MCP Task Completion Verbose Logs

**************************************************

Tasks:
[
    {
        "task": "Customer C-1003 wants to know whether order ORD-9003 can be refunded.\n",
        "steps_taken": [
            "Tools called by agent: \n\n<Tool Called>\n\n**This does not appear to user**\nName: 
get_order_status\nArgs: {'order_id': 'ORD-9003'}\nResult: \n{'order_id': 'ORD-9003', 'customer_id': 'C-1003', 
'status': 'delivered', 'item': 'Smartwatch', 'amount': 149.0, 'delivered_days_ago': 42, 'issue': 'Customer asked 
about refund eligibility.'}\n</Tool Called>\n\n<Tool Called>\n\n**This does not appear to user**\nName: 
search_policy_docs\nArgs: {'query': 'refund policy return window delivered days eligibility delivered_days_ago 42 
smartwatch'}\nResult: \n{'query': 'refund policy return window delivered days eligibility delivered_days_ago 42 
smartwatch', 'matches': [{'policy_id': 'refund', 'title': 'Refund policy', 'summary': 'Most delivered items are 
refundable within 30 days. Damaged items can be escalated after 30 days for manual review.'}, {'policy_id': 
'delivery', 'title': 'Delivery delay policy', 'summary': 'If an order is delayed, share the reason and estimated 
delivery date. Shipping fees may be refunded for delays over 5 days.'}, {'policy_id': 'damaged_item', 'title': 
'Damaged item policy', 'summary': 'Damaged delivered items should be reviewed with order details. Create a 
high-priority ticket for expensive or urgent damaged-item cases.'}, {'policy_id': 'cancellation', 'title': 
'Cancellation policy', 'summary': 'Cancelled orders should be explained using the cancellation reason. Ask the 
customer to retry payment or contact billing when payment authorization fails.'}, {'policy_id': 'shipping_status', 
'title': 'Shipping status policy', 'summary': 'For shipped or processing orders, share the current status and 
estimated delivery date when available.'}]}\n</Tool Called>\n",
            "Agent's response to user: \nHere\u2019s what I found for order ORD-9003 (Smartwatch) for customer 
C-1003: it was delivered 42 days ago. Our refund policy allows most delivered items to be refunded within 30 days. 
Since this is beyond 30 days, it isn\u2019t eligible for a standard refund.\n\nIf the smartwatch arrived damaged or
defective, we can submit it for a manual review even after 30 days. If that\u2019s the case, please share a brief 
description of the issue and a few photos showing the problem, and I can open a high-priority ticket for 
you.\n\nNext best step: let me know if the item is damaged/defective and if you\u2019d like me to create a review 
ticket. If not, I\u2019m afraid we won\u2019t be able to process a refund for this order."
        ]
    },
    {
        "task": "Thank you.\n",
        "steps_taken": [
            "Agent's response to user: \nYou are welcome. I hope the information provided was helpful."
        ]
    }
] 
 
Individual Scores & Reasons:
[(1.0, 'The agent clearly answered the refund eligibility: ORD-9003 is beyond the 30-day window and not eligible 
for a standard refund, with an exception path for damaged/defective items and actionable next steps. This directly 
addresses whether the order can be refunded.'), (1.0, "The agent courteously acknowledged the user's thanks; no 
further action was requested or needed.")]
 
Score: 1.0

...

MCP Task Completion Verbose Logs

**************************************************

Tasks:
[
    {
        "task": "Customer C-1004 is travelling soon and asks when the ergonomic keyboard will arrive.\n",
        "steps_taken": [
            "Tools called by agent: \n\n<Tool Called>\n\n**This does not appear to user**\nName: 
get_customer_profile\nArgs: {'customer_id': 'C-1004'}\nResult: \n{'customer_id': 'C-1004', 'name': 'Dev Kapoor', 
'tier': 'Gold', 'primary_order_id': 'ORD-9004', 'email': 'dev@example.com'}\n</Tool Called>\n\n<Tool 
Called>\n\n**This does not appear to user**\nName: get_order_status\nArgs: {'order_id': 'ORD-9004'}\nResult: 
\n{'order_id': 'ORD-9004', 'customer_id': 'C-1004', 'status': 'processing', 'item': 'Ergonomic keyboard', 'amount':
89.0, 'estimated_delivery': '2026-07-16', 'issue': 'Customer wants delivery expectations before travel.'}\n</Tool 
Called>\n",
            "Agent's response to user: \nI checked your order. The ergonomic keyboard (order ORD-9004) is currently
processing and is estimated to be delivered on 2026-07-16.\n\nIf that timing conflicts with your travel, I can look
into available options (such as updating the delivery address or arranging a carrier hold, if eligible). Next best 
step: let me know your departure date and whether you want me to check delivery-change options or create a support 
ticket on your behalf."
        ]
    },
    {
        "task": "Thank you.\n",
        "steps_taken": [
            "Agent's response to user: \nYou are welcome. I hope the information provided was helpful."
        ]
    }
] 
 
Individual Scores & Reasons:
[(1.0, 'The agent clearly answered the question by providing the estimated delivery date (2026-07-16) for the 
ergonomic keyboard and offered relevant next steps. This fully addresses the user’s request about when it will 
arrive.'), (1.0, "The agent politely acknowledged the user's thanks, which appropriately completes the interaction 
without missing any requested action.")]
 
Score: 1.0

======================================================================

,case_id,expected_label,metric,score,success,reason
0,CASE-002-20260715201406,Positive,MCPTaskCompletionMetric,1.0,True,"The score is 1.0 because the agent accurately completed the task by correctly determining refund eligibility for ORD-9003, clearly outlining the exception path and actionable next steps, and then appropriately handling the follow-up by courteously acknowledging the user’s thanks without unnecessary actions, demonstrating consistent, correct behavior across interactions and thus passing the metric."
1,CASE-008-20260715201443,Positive,MCPTaskCompletionMetric,1.0,True,"The score is 1.0 because the agent accurately fulfilled the user’s request by providing the specific estimated delivery date for the ergonomic keyboard and offering relevant next steps, then appropriately acknowledged the user’s thanks to close the interaction, demonstrating complete and context-aware task execution without omissions."


## 14. Final Report

In [ ]:
final_report = pd.concat(
    [mcp_use_report, multi_turn_use_report, mcp_task_completion_report],
    ignore_index=True,
)

final_report

,case_id,expected_label,metric,score,success,reason
0,CASE-002-20260715201406,Positive,MCPUseMetric,0.750,True,"[\n\tThe agent correctly used get_order_status to verify the order’s delivery status and timing, and search_policy_docs to confirm the refund policy and exceptions—both relevant to assessing eligibility. They avoided unnecessary calls like get_customer_profile and didn’t prematurely create a ticket. However, the available calculate_refund tool is specifically designed to determine refund eligibility for an order and would have been a more direct and potentially more accurate choice. Not leveraging calculate_refund represents a missed, more appropriate tool for the task.\n\tBoth tool calls used well-formed arguments that matched the tools’ input schemas and were suitable for the task. get_order_status received the required 'order_id' as a string with no extraneous fields. search_policy_docs received a single 'query' string relevant to refund eligibility and timing; no additional properties were included. No required arguments were missing or malformed, and the contents were aligned with the user’s request.\n]\n"
1,CASE-008-20260715201443,Positive,MCPUseMetric,1.000,True,"[\n\tThe agent chose and used the correct tools for the task. They first called get_customer_profile to retrieve the customer’s primary order ID since the user did not provide an order ID, then used get_order_status to obtain the delivery estimate for the ergonomic keyboard. This sequence directly addresses the user’s request. No unnecessary tools were called, and other available tools (search_policy_docs, calculate_refund, create_support_ticket) were appropriately not used at this stage. The tool usage was appropriate and sufficient for answering the delivery timing question.\n\tBoth tool calls used correct and well-formed arguments that matched each tool’s schema and the user’s task. get_customer_profile received {'customer_id': 'C-1004'} (string, required field only, no extras) and get_order_status received {'order_id': 'ORD-9004'} (string, required field only, no extras). The contents were semantically appropriate for determining delivery timing of the ergonomic keyboard associated with the customer’s order. No required fields were missing or malformed, and no extraneous properties were included.\n]\n"
2,CASE-002-20260715201406,Positive,MultiTurnMCPUseMetric,0.875,True,"The metric effectively passed, as reflected by a strong final score of 0.875, driven by consistent, appropriate tool use, correct argument schemas, and prudent avoidance of unnecessary calls in simpler interactions. The primary shortcoming was a missed opportunity to invoke the calculate_refund tool when assessing refund eligibility, which would have provided a more authoritative determination despite the agent’s otherwise correct use of get_order_status and search_policy_docs. Overall, tool usage was accurate and well-justified, with no misuses observed, and the single gap did not outweigh the generally strong performance."
3,CASE-008-20260715201443,Positive,MultiTurnMCPUseMetric,1.000,True,"The score is 1.0 because across interactions the agent consistently chose the correct tools when needed, avoided unnecessary calls for simple tasks, and executed all invocations with correctly formatted arguments that matched the required schemas. It accurately used get_customer_profile to obtain the order ID and get_order_status to retrieve the ETA, offered appropriate options without asserting unsupported policies, and refrained from tool usage for a simple acknowledgment. With no schema violations or missed opportunities for better tool use, performance met all criteria, justifying a perfect score."
4,CASE-002-20260715201406,Positive,MCPTaskCompletionMetric,1.000,True,"The metric passed with a perfect score (1.0) because the agent accurately applied the 30-day refund policy, correctly determined the order was ineligible while noting the damaged/defective exception with clear next steps, and

## 15. Export Final Report

This cell writes the metric results to Excel. In Colab, it also starts a browser download.

In [ ]:
REPORT_FILE = Path("DeepEval_MCP_Final_Report.xlsx")

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    final_report.to_excel(writer, sheet_name="final_report", index=False)
    mcp_use_report.to_excel(writer, sheet_name="mcp_use_metric", index=False)
    multi_turn_use_report.to_excel(writer, sheet_name="multi_turn_mcp_use", index=False)
    mcp_task_completion_report.to_excel(writer, sheet_name="mcp_task_completion", index=False)

print(f"Saved report to {REPORT_FILE}")

try:
    from google.colab import files

    files.download(str(REPORT_FILE))
except ImportError:
    from IPython.display import FileLink, display

    display(FileLink(str(REPORT_FILE)))

Saved report to DeepEval_MCP_Final_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>